# SaludAI — Getting Started

This notebook demonstrates the core components of SaludAI:

1. **FHIR Client** — Connect to a FHIR server, search and read resources
2. **Terminology Resolver** — Resolve clinical terms to SNOMED CT, ICD-10, LOINC codes
3. **Query Builder** — Build complex FHIR queries with a fluent API
4. **Locale Packs** — Adapt the agent to different health systems (Argentina, US)

## Prerequisites

```bash
docker compose up -d    # Start HAPI FHIR with synthetic patient data
# Wait ~30 seconds for data to load
```

## 1. FHIR Client

`FHIRClient` is an async client for interacting with any FHIR R4 server.

In [1]:
from saludai_core.fhir_client import FHIRClient

# Connect to local FHIR server (reads SALUDAI_FHIR_SERVER_URL from .env)
async with FHIRClient() as client:
    capability = await client.check_connection()
    print(f"FHIR server connected")
    print(f"  Version: {capability.fhirVersion}")
    print(f"  Software: {capability.software.name if capability.software else 'N/A'}")

2026-04-06 01:34:15 [info     ] fhir.check_connection          fhir_server=http://localhost:8890/fhir
2026-04-06 01:34:15 [info     ] fhir.connected                 fhir_server=http://localhost:8890/fhir fhir_version=4.0.1
FHIR server connected
  Version: 4.0.1
  Software: HAPI FHIR Server
2026-04-06 01:34:15 [debug    ] fhir.client_closed             fhir_server=http://localhost:8890/fhir


### Search patients

`search()` returns a FHIR Bundle as a dictionary. You can search by name, state, or any FHIR parameter.

In [2]:
async with FHIRClient() as client:
    bundle = await client.search("Patient", {"_count": "5"})

    print(f"Total patients in system: {bundle.get('total', 'N/A')}")
    print(f"Showing: {len(bundle.get('entry', []))} patients\n")

    for entry in bundle.get("entry", []):
        patient = entry["resource"]
        name = patient.get("name", [{}])[0]
        given = " ".join(name.get("given", []))
        family = name.get("family", "")
        birth = patient.get("birthDate", "N/A")
        gender = patient.get("gender", "N/A")
        print(f"  {given} {family} | Born: {birth} | Gender: {gender}")

2026-04-06 01:34:34 [info     ] fhir.search                    fhir_server=http://localhost:8890/fhir params={'_count': '5'} resource_type=Patient
Total patients in system: N/A
Showing: 5 patients

  Juan Castro | Born: 2007-06-28 | Gender: male
  Andrés Castro | Born: 1967-01-12 | Gender: male
  Ana González | Born: 1974-09-02 | Gender: female
  Héctor Martínez | Born: 2000-03-25 | Gender: male
  Andrea García | Born: 1951-12-07 | Gender: female
2026-04-06 01:34:34 [debug    ] fhir.client_closed             fhir_server=http://localhost:8890/fhir


### Search clinical conditions

Search conditions by SNOMED CT code and follow references to patients using `_include`.

In [3]:
async with FHIRClient() as client:
    # Search for type 2 diabetes conditions (SNOMED CT: 44054006)
    bundle = await client.search(
        "Condition",
        {
            "code": "http://snomed.info/sct|44054006",
            "_count": "10",
            "_include": "Condition:subject",  # Include referenced patients
        },
    )

    conditions = []
    patients = {}

    for entry in bundle.get("entry", []):
        resource = entry["resource"]
        if resource["resourceType"] == "Patient":
            patients[f"Patient/{resource['id']}"] = resource
        else:
            conditions.append(resource)

    print(f"Type 2 diabetes conditions found: {len(conditions)}")
    print(f"Patients included via _include: {len(patients)}\n")

    for cond in conditions[:5]:
        subject_ref = cond.get("subject", {}).get("reference", "")
        patient = patients.get(subject_ref, {})
        name = patient.get("name", [{}])[0]
        patient_name = " ".join(name.get("given", [])) + " " + name.get("family", "")
        onset = cond.get("onsetDateTime", "N/A")[:10] if cond.get("onsetDateTime") else "N/A"
        print(f"  Patient: {patient_name.strip()} | Onset: {onset}")

2026-04-06 01:35:12 [info     ] fhir.search                    fhir_server=http://localhost:8890/fhir params={'code': 'http://snomed.info/sct|44054006', '_count': '10', '_include': 'Condition:subject'} resource_type=Condition
Type 2 diabetes conditions found: 10
Patients included via _include: 8

  Patient: Lucía Gómez | Onset: 2003-05-30
  Patient: Agustín Álvarez | Onset: 1987-12-16
  Patient: Romina Ríos | Onset: 1988-11-30
  Patient: Victoria González | Onset: 1999-12-15
  Patient: Andrés Juárez | Onset: 1969-09-26
2026-04-06 01:35:13 [debug    ] fhir.client_closed             fhir_server=http://localhost:8890/fhir


### Read an individual resource

`read()` fetches a FHIR resource by ID and returns it as a `fhir.resources` model.

In [4]:
async with FHIRClient() as client:
    bundle = await client.search("Patient", {"_count": "1"})
    patient_id = bundle["entry"][0]["resource"]["id"]

    # Read full resource (returns fhir.resources model)
    patient = await client.read("Patient", patient_id)
    print(f"Type: {patient.get_resource_type()}")
    print(f"ID: {patient.id}")
    print(f"Name: {patient.name[0].given[0]} {patient.name[0].family}")
    print(f"Birth date: {patient.birthDate}")
    print(f"Gender: {patient.gender}")

    if patient.address:
        addr = patient.address[0]
        print(f"Address: {addr.city}, {addr.state}, {addr.country}")

2026-04-06 01:35:30 [info     ] fhir.search                    fhir_server=http://localhost:8890/fhir params={'_count': '1'} resource_type=Patient
2026-04-06 01:35:30 [info     ] fhir.read                      fhir_server=http://localhost:8890/fhir resource_id=1000 resource_type=Patient
Type: Patient
ID: 1000
Name: Juan Castro
Birth date: 2007-06-28
Gender: male
Address: None, Buenos Aires, AR
2026-04-06 01:35:30 [debug    ] fhir.client_closed             fhir_server=http://localhost:8890/fhir


## 2. Terminology Resolver

`TerminologyResolver` maps clinical terms to standard codes. It supports exact matches, colloquial aliases, and fuzzy matching.

The resolver is locale-aware — it loads terminology from the active locale pack.

In [5]:
from saludai_core.locales import load_locale_pack
from saludai_core.terminology import TerminologyResolver

# Try both locales — change to "ar" for Spanish
locale = load_locale_pack("en_us")
resolver = TerminologyResolver(locale_pack=locale)

print(f"Locale: {locale.name} ({locale.language})")
print(f"Terminology systems: {[s.key for s in locale.terminology_systems]}")
print(f"Total concepts: {resolver.concept_count}")

2026-04-06 01:35:58 [debug    ] terminology_csv_loaded         concepts=114 filename=snomed_en_us.csv system=SNOMED_CT
2026-04-06 01:35:58 [debug    ] terminology_csv_loaded         concepts=45 filename=icd10_cm.csv system=ICD_10_CM
2026-04-06 01:35:58 [debug    ] terminology_csv_loaded         concepts=37 filename=loinc_en.csv system=LOINC
2026-04-06 01:35:58 [debug    ] terminology_csv_loaded         concepts=13 filename=atc_en.csv system=ATC
2026-04-06 01:35:58 [info     ] terminology_resolver_initialized systems={'SNOMED_CT': 114, 'CIE_10': 0, 'ICD_10_CM': 45, 'LOINC': 37, 'ATC': 13} total_concepts=209
Locale: United States (en)
Terminology systems: ['snomed_ct', 'icd_10_cm', 'loinc', 'atc']
Total concepts: 209


### Resolve clinical terms

`resolve()` finds the best match for a term. Works with formal names, colloquial aliases, and abbreviations.

In [6]:
# English terms (en_us locale)
terms = [
    "type 2 diabetes",
    "high blood pressure",
    "COPD",
    "HbA1c",               # LOINC lab test
    "metformin",            # ATC medication
    "E11",                  # ICD-10-CM code
]

# Spanish terms — uncomment and switch locale to "ar":
# terms = [
#     "diabetes tipo 2",
#     "presión alta",
#     "asma",
#     "glucosa en sangre",
#     "metformina",
#     "E11",
# ]

for term in terms:
    match = resolver.resolve(term)
    if match.concept:
        print(f"  '{term}'")
        print(f"    -> {match.concept.display} ({match.concept.system.value})")
        print(f"    -> Code: {match.concept.code} | Score: {match.score:.0f} | Type: {match.match_type.value}")
        print()

2026-04-06 01:36:29 [info     ] terminology_resolved           code=44054006 match_type=exact_alias score=100.0 term='type 2 diabetes'
  'type 2 diabetes'
    -> Type 2 diabetes mellitus (http://snomed.info/sct)
    -> Code: 44054006 | Score: 100 | Type: exact_alias

2026-04-06 01:36:29 [info     ] terminology_resolved           code=59621000 match_type=exact_alias score=100.0 term='high blood pressure'
  'high blood pressure'
    -> Essential hypertension (http://snomed.info/sct)
    -> Code: 59621000 | Score: 100 | Type: exact_alias

2026-04-06 01:36:29 [info     ] terminology_resolved           code=13645005 match_type=exact_display score=100.0 term=COPD
  'COPD'
    -> Chronic obstructive pulmonary disease (http://snomed.info/sct)
    -> Code: 13645005 | Score: 100 | Type: exact_display

2026-04-06 01:36:29 [info     ] terminology_resolved           code=4548-4 match_type=exact_alias score=100.0 term=HbA1c
  'HbA1c'
    -> Hemoglobin A1c (http://loinc.org)
    -> Code: 4548-4 | Sco

### Search for multiple matches

`search()` returns the top matches ranked by score — useful for ambiguous terms.

In [7]:
matches = resolver.search("diabetes", max_results=5)

print(f"Results for 'diabetes' ({len(matches)} found):\n")
for m in matches:
    confident = "OK" if m.is_confident else "REVIEW"
    print(f"  [{confident}] {m.concept.display}")
    print(f"        Code: {m.concept.code} | System: {m.concept.system.value} | Score: {m.score:.1f}")
    print()

Results for 'diabetes' (5 found):

  [OK] Type 2 diabetes mellitus
        Code: 44054006 | System: http://snomed.info/sct | Score: 81.7

  [OK] Type 1 diabetes mellitus
        Code: 73211009 | System: http://snomed.info/sct | Score: 81.7

  [OK] Type 2 diabetes mellitus
        Code: E11 | System: http://hl7.org/fhir/sid/icd-10-cm | Score: 81.7

  [OK] Type 1 diabetes mellitus
        Code: E10 | System: http://hl7.org/fhir/sid/icd-10-cm | Score: 81.7

  [OK] Gestational diabetes
        Code: 11687002 | System: http://snomed.info/sct | Score: 76.9



## 3. Query Builder

`FHIRQueryBuilder` builds complex FHIR queries with a fluent API — no manual string manipulation.

In [ ]:
from saludai_core.query_builder import (
    FHIRQueryBuilder,
    SortOrder,
    SummaryMode,
    snomed,
)

# Build a complex query:
# "Active hypertension conditions, including patient data, sorted by date"
query = (
    FHIRQueryBuilder("Condition")
    .where("code", snomed("59621000"))          # Essential hypertension
    .where_string("clinical-status", "active")   # Active only
    .include("subject")                          # Include referenced Patient
    .count(50)
    .sort("onset-date", SortOrder.DESC)          # Most recent first
    .build()
)

params = query.to_params()
print(f"Resource type: {query.resource_type}")
print(f"Generated parameters:")
for key, value in params.items():
    print(f"  {key} = {value}")

### Execute the query against FHIR

Combine QueryBuilder with FHIRClient to run the search.

In [ ]:
async with FHIRClient() as client:
    bundle = await client.search(query.resource_type, params)

    conditions = [
        e["resource"] for e in bundle.get("entry", [])
        if e["resource"]["resourceType"] == "Condition"
    ]
    patients = {
        f"Patient/{e['resource']['id']}": e["resource"]
        for e in bundle.get("entry", [])
        if e["resource"]["resourceType"] == "Patient"
    }

    print(f"Patients with active hypertension: {len(conditions)}\n")

    for cond in conditions[:5]:
        ref = cond.get("subject", {}).get("reference", "")
        patient = patients.get(ref, {})
        name = patient.get("name", [{}])[0]
        patient_name = " ".join(name.get("given", [])) + " " + name.get("family", "")
        code_display = cond.get("code", {}).get("coding", [{}])[0].get("display", "N/A")
        print(f"  {patient_name.strip()} — {code_display}")

### Count resources with _summary=count

Get totals without downloading data using `SummaryMode.COUNT`.

In [ ]:
async with FHIRClient() as client:
    resource_types = [
        "Patient", "Condition", "Observation", "MedicationRequest",
        "Encounter", "Procedure", "DiagnosticReport", "AllergyIntolerance",
        "Immunization", "CarePlan",
    ]

    print("Resources in FHIR server:\n")
    for rt in resource_types:
        count_query = FHIRQueryBuilder(rt).summary(SummaryMode.COUNT).build()
        bundle = await client.search(rt, count_query.to_params())
        total = bundle.get("total", "?")
        print(f"  {rt}: {total}")

## 4. Locale Packs

Locale packs adapt SaludAI to different countries and health systems. Compare Argentina (`ar`) and United States (`en_us`):

In [8]:
from saludai_core.locales import available_locales, load_locale_pack

print(f"Available locales: {available_locales()}\n")

for code in available_locales():
    pack = load_locale_pack(code)
    print(f"{'=' * 40}")
    print(f"Locale: {pack.name} ({pack.code})")
    print(f"  Language: {pack.language}")
    print(f"  Terminology systems: {[s.key for s in pack.terminology_systems]}")
    print(f"  FHIR profiles: {len(pack.fhir_profiles)}")
    print(f"  Extensions: {len(pack.extensions)}")
    print(f"  Identifier systems: {[i.name for i in pack.identifier_systems]}")
    print()

Available locales: ['ar', 'en_us']

Locale: Argentina (ar)
  Language: es
  Terminology systems: ['snomed_ct', 'cie_10', 'loinc', 'atc']
  FHIR profiles: 6
  Extensions: 7
  Identifier systems: ['DNI', 'REFEPS', 'REFES']

Locale: United States (en_us)
  Language: en
  Terminology systems: ['snomed_ct', 'icd_10_cm', 'loinc', 'atc']
  FHIR profiles: 9
  Extensions: 3
  Identifier systems: ['SSN', 'NPI', 'MRN']



---

**Next:** In [notebook 02](02-agent-queries.ipynb) we see how the agent combines these components to answer clinical questions in natural language.